# Week 8 Lab - Spatial Indexing and Spatial Queries

This notebook follows the same workflow as the Week 8 lecture. **Part A** repeats the six-document `castles near Vienna` example from class. **Part B** applies the same workflow to a larger supplied Vienna POI dataset. **Part C** asks you to interpret the results you produced.

Use the course Moodle JupyterHub. All data needed for this lab are included in the notebook, so no separate data file is required.


## Before you start

Work from top to bottom and keep your outputs visible when you save the notebook.

- **Part A - Guided:** run the supplied code and compare the outputs with the lecture.
- **Part B - Independent:** repeat the same workflow on a different dataset. Only a few short code cells are left for you to complete.
- **Part C - Interpretation:** answer three short questions using your own Part B results.
- Cells labelled **Optional visualization** are there only to help you see what the code is doing. They are **not required for submission**.

The six lecture points use the same simplified kilometre grid as the slides. The larger POI dataset is already supplied with metre-based coordinates so that you can focus on the spatial-query workflow rather than coordinate conversion.


## Setup


In [ ]:
import pandas as pd
import geopandas as gpd
import spacy
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from shapely.geometry import Point, box
from io import StringIO
from IPython.display import display

# We use a blank English spaCy pipeline only for simple tokenisation.
nlp = spacy.blank("en")


# Part A - Guided Lab: the six lecture documents


## A1. Build the same six-document collection

This is the same A-F collection used in the lecture. Each document has an ID, a short description and one point location.


In [ ]:
# Create the six documents from the lecture.
places = gpd.GeoDataFrame([
    ("A", "A guide to Castle A.", Point(6.25, 6.25)),
    ("B", "A guide to Castle B.", Point(13.75, 3.75)),
    ("C", "Visitor information for the library.", Point(8.75, 10)),
    ("D", "Visitor information for the sports field.", Point(17.5, 15)),
    ("E", "A guide to Castle E.", Point(8.75, 13.75)),
    ("F", "A guide to the walking trail.", Point(2.5, 16.25)),
], columns=["place_id", "description", "geometry"], geometry="geometry")


In [ ]:
# Use the same Vienna reference point and the same first definition of "near".
vienna = Point(8.75, 8.75)
radius = 5  # km in the simplified lecture grid

# Inspect the collection before querying it.
display(places)


**Expected:** six documents, A-F.


### Optional visualization - see the collection and the 5 km query

This picture is only for understanding. It is **not required for submission**.


In [ ]:
# Start the optional figure with the six document locations.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(places.geometry.x, places.geometry.y, label="Documents")

# Add the document IDs next to their points.
for _, row in places.iterrows():
    ax.annotate(row["place_id"], (row.geometry.x, row.geometry.y),
                xytext=(5, 5), textcoords="offset points")

# Add the Vienna query point and the exact 5 km circle.
ax.scatter([vienna.x], [vienna.y], marker="x", s=80, label="Vienna reference point")
ax.annotate("V", (vienna.x, vienna.y), xytext=(5, 5), textcoords="offset points")
ax.add_patch(Circle((vienna.x, vienna.y), radius, fill=False, linewidth=2, label="5 km radius"))

# Finish the optional figure.
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("Lecture example: six documents and the 5 km query")
ax.set_aspect("equal")
ax.legend()
plt.show()


## A2. Apply the spatial condition directly

First repeat the direct distance scan from the lecture.


In [ ]:
# Calculate one distance for every document location.
distance = places.distance(vienna)

# Put the distances beside the document IDs so they are easy to read.
distance_table = places[["place_id"]].copy()
distance_table["distance_km"] = distance.round(2)
display(distance_table)


In [ ]:
# Keep only documents whose distance is at most 5 km.
nearby = places[distance <= radius]

# Store only their document IDs; we will combine IDs with the text result later.
spatial_ids = set(nearby["place_id"])
print("Spatial IDs:", sorted(spatial_ids))


**Expected:** the 5 km spatial matches are **A, C and E**.


## A3. Apply the text condition

Now repeat the small inverted-index step from the lecture.


In [ ]:
# Start an empty inverted index: term -> set of document IDs.
index = {}

# Read every document description and add each term to the index.
for _, row in places.iterrows():
    for token in nlp(row["description"]):
        # Ignore spaces and punctuation because they are not useful lookup terms.
        if token.is_punct or token.is_space:
            continue

        # Lowercase the term so "Castle" and "castle" use the same key.
        term = token.text.lower()
        index.setdefault(term, set()).add(row["place_id"])


In [ ]:
# The query says "castles", while the documents use "castle".
# Look up both forms and combine their postings.
castle_ids = (
    index.get("castle", set())
    | index.get("castles", set())
)
print("Castle IDs:", sorted(castle_ids))


### Combine the text and spatial conditions


In [ ]:
# A final GIR match must satisfy BOTH conditions.
result_ids = spatial_ids & castle_ids
print("Final GIR IDs:", sorted(result_ids))


In [ ]:
# Use the IDs to retrieve the original descriptions.
hits = places[places["place_id"].isin(result_ids)]
display(hits[["place_id", "description"]])


**Expected:** text matches = **A, B, E**; final GIR matches = **A, E**.


## A4. Find candidates with a search box

The 5 km circle fits inside a square that extends 5 km in x and y from Vienna. The square is only a **candidate filter**.


In [ ]:
# Read the centre coordinates once so the box limits are easy to construct.
x, y = vienna.x, vienna.y

# Check the x range for every point.
x_ok = places.geometry.x.between(x - radius, x + radius)

# Check the y range for every point.
y_ok = places.geometry.y.between(y - radius, y + radius)


In [ ]:
# A point must pass BOTH coordinate-range checks to become a box candidate.
box_candidates = places[x_ok & y_ok].copy()
display(box_candidates[["place_id", "description"]])


**Expected:** the search-box candidates are **A, B, C and E**.


### Optional visualization - compare the circle and its search box

This makes it easier to see why B is a box candidate but not a final 5 km match. It is **not required for submission**.


In [ ]:
# Start with the six document points and Vienna.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(places.geometry.x, places.geometry.y, label="Documents")
for _, row in places.iterrows():
    ax.annotate(row["place_id"], (row.geometry.x, row.geometry.y),
                xytext=(5, 5), textcoords="offset points")
ax.scatter([vienna.x], [vienna.y], marker="x", s=80, label="Vienna reference point")

# Add the exact circle and the larger candidate box.
ax.add_patch(Circle((x, y), radius, fill=False, linewidth=2, label="5 km radius"))
ax.add_patch(Rectangle(
    (x-radius, y-radius), 2*radius, 2*radius,
    fill=False, linestyle="--", linewidth=2, label="Search box"
))

# Finish the optional comparison figure.
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("Search box candidates vs exact 5 km distance")
ax.set_aspect("equal")
ax.legend()
plt.show()


### Check the exact distance for the candidates


In [ ]:
# Calculate distances only for the box candidates.
candidate_distance = box_candidates.geometry.distance(vienna)
box_candidates["distance_km"] = candidate_distance

display(
    box_candidates[["place_id", "distance_km"]]
    .round({"distance_km": 2})
)


In [ ]:
# Apply the real 5 km condition after the candidate step.
box_nearby = box_candidates[box_candidates["distance_km"] <= radius]
box_spatial_ids = set(box_nearby["place_id"])
print("Spatial IDs after exact distance:", sorted(box_spatial_ids))


**Expected:** B is in the square but is about **7.07 km** away, so the exact result is again **A, C and E**.


## A5. Build and query the spatial index

The lecture used group boxes and STR to explain the R-tree. In practice, GeoPandas builds the spatial index for us through `.sindex`.


In [ ]:
# Build the spatial index ONCE for the document collection.
place_index = places.sindex

# Build the same 5 km search box as before.
search_box = box(
    x - radius,
    y - radius,
    x + radius,
    y + radius,
)


In [ ]:
# Query the index with the search box.
# The result contains row positions, not document IDs.
positions = place_index.query(search_box, sort=True)
print("Candidate row positions:", positions.tolist())


In [ ]:
# Use iloc to retrieve the complete candidate records from those row positions.
index_candidates = places.iloc[positions].copy()
display(index_candidates[["place_id", "description"]])


### Reuse the exact-distance step


In [ ]:
# The index gives box candidates, so we still check the real distance.
index_candidates["distance_km"] = index_candidates.geometry.distance(vienna)
index_nearby = index_candidates[index_candidates["distance_km"] <= radius]
index_spatial_ids = set(index_nearby["place_id"])
print("Spatial IDs:", sorted(index_spatial_ids))


In [ ]:
# Combine the indexed spatial result with the same castle text result.
index_result_ids = index_spatial_ids & castle_ids
index_hits = places[places["place_id"].isin(index_result_ids)]
display(index_hits[["place_id", "description"]])


**Compare:** the spatial index returns the same box candidates, and the final result is still **A and E**.


## A6. Change the definition of “near” to 7.5 km

The documents have not changed, so **do not rebuild `place_index`**. Only the query changes.


In [ ]:
# Change only the radius used by the new query.
radius_2 = 7.5

# Build a larger search box around the same Vienna point.
search_box_2 = box(
    x - radius_2,
    y - radius_2,
    x + radius_2,
    y + radius_2,
)


In [ ]:
# Reuse the SAME spatial index with the new box.
positions_2 = place_index.query(search_box_2, sort=True)
candidates_2 = places.iloc[positions_2].copy()
print("7.5 km box candidates:", candidates_2["place_id"].tolist())


In [ ]:
# Check the exact 7.5 km distances for the new candidate set.
candidates_2["distance_km"] = candidates_2.geometry.distance(vienna)
nearby_2 = candidates_2[candidates_2["distance_km"] <= radius_2]
spatial_ids_2 = set(nearby_2["place_id"])

display(
    candidates_2[["place_id", "distance_km"]]
    .sort_values("distance_km")
    .round({"distance_km": 2})
)
print("7.5 km spatial IDs:", sorted(spatial_ids_2))


In [ ]:
# The text condition did not change, so reuse castle_ids too.
result_ids_2 = spatial_ids_2 & castle_ids
print("7.5 km final GIR IDs:", sorted(result_ids_2))


**Expected:** box candidates = **A, B, C, E, F**; exact spatial matches = **A, B, C, E**; final GIR matches = **A, B, E**.


### Optional visualization - compare 5 km and 7.5 km

This visualization shows what changed when the definition of **near** changed. The stored document collection and its spatial index did not change. It is **not required for submission**.


In [ ]:
# Plot the documents and the common query point.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(places.geometry.x, places.geometry.y, label="Documents")
for _, row in places.iterrows():
    ax.annotate(row["place_id"], (row.geometry.x, row.geometry.y),
                xytext=(5, 5), textcoords="offset points")
ax.scatter([vienna.x], [vienna.y], marker="x", s=80, label="Vienna reference point")

# Draw both radius definitions around the same point.
ax.add_patch(Circle((x, y), radius, fill=False, linewidth=2, label="5 km"))
ax.add_patch(Circle((x, y), radius_2, fill=False, linestyle="--", linewidth=2, label="7.5 km"))

# Finish the optional comparison figure.
ax.set_xlabel("x (km)")
ax.set_ylabel("y (km)")
ax.set_title("Changing only the query radius")
ax.set_aspect("equal")
ax.legend()
plt.show()


## A7. Guided result recap

Use this recap as a simple visible result check. Compare these expected results with the outputs you can see above:

- 5 km spatial IDs: **A, C, E**
- castle text IDs: **A, B, E**
- 5 km final GIR IDs: **A, E**
- indexed 5 km box candidates: **A, B, C, E**
- 7.5 km final GIR IDs: **A, B, E**

If your visible outputs differ, go back to the relevant step and compare it with the lecture example.


## From the guided example to the larger dataset

You have now repeated the complete lecture workflow on the six documents. In Part B, the dataset changes, but the method stays the same:

**topic matches → search box → spatial index → exact distance → combine IDs → change the radius and reuse the index**

Use Part A as your reference. The supplied POI coordinates are already in metres, so you can use radii such as 500 and 750 directly.


# Part B - Independent Practice: 80 Vienna POIs

The supplied table contains **80 Vienna-area practice POIs** with stable IDs, names, categories and metre-based coordinates. The data preparation is already done for you.

Your query is:

> **cafés near the query point**

First define **near = within 500 m**. Then repeat the query with **750 m** while reusing the same spatial index.

Only the small cells marked **Your code** need to be completed. Use Part A as a model.


In [ ]:
# Supplied practice data. You do not need to edit this cell.
POI_CSV = """poi_id,name,category,x_m,y_m
VIE001,Restaurant 1,restaurant,600876.14,5340694.13
VIE002,University 2,university,600838.79,5341058.92
VIE003,Tram_Stop 3,tram_stop,600957.65,5341137.45
VIE004,Museum 4,museum,600811.32,5340803.46
VIE005,Library 5,library,601095.46,5341063.98
VIE006,Bookshop 6,bookshop,600885.08,5341136.32
VIE007,Cafe 7,cafe,601121.1,5340989.8
VIE008,Tram_Stop 8,tram_stop,600874.65,5340871.9
VIE009,Cafe 9,cafe,601475.75,5340282.63
VIE010,Hotel 10,hotel,600614.97,5340024.88
VIE011,Museum 11,museum,601002.88,5340132.25
VIE012,Cafe 12,cafe,601321.4,5341572.02
VIE013,Bookshop 13,bookshop,601390.52,5341389.13
VIE014,Bookshop 14,bookshop,600612.11,5341186.1
VIE015,Museum 15,museum,601738.93,5340411.38
VIE016,Park 16,park,601556.57,5341616.65
VIE017,Library 17,library,601412.56,5340381.07
VIE018,Park 18,park,601037.61,5340976.11
VIE019,Restaurant 19,restaurant,600639.87,5341014.69
VIE020,Bakery 20,bakery,600552.8,5341026.69
VIE021,Library 21,library,600689.46,5340498.07
VIE022,Bakery 22,bakery,600934.74,5340284.51
VIE023,Hotel 23,hotel,600766.14,5341296.71
VIE024,Restaurant 24,restaurant,600177.09,5341430.85
VIE025,Bakery 25,bakery,600464.68,5340226.32
VIE026,University 26,university,601676.29,5340785.6
VIE027,Park 27,park,600917.49,5340586.95
VIE028,Tram_Stop 28,tram_stop,600890.06,5341177.23
VIE029,Restaurant 29,restaurant,600957.93,5340619.22
VIE030,Cafe 30,cafe,601872.49,5340416.22
VIE031,Hotel 31,hotel,601466.34,5341254.24
VIE032,Bookshop 32,bookshop,600455.06,5340455.35
VIE033,Park 33,park,601753.56,5340796.05
VIE034,Library 34,library,601863.38,5340930.14
VIE035,University 35,university,600632.03,5340893.31
VIE036,Park 36,park,600740.57,5341273.87
VIE037,Restaurant 37,restaurant,600516.42,5340145.4
VIE038,Museum 38,museum,601025.19,5340259.25
VIE039,Hotel 39,hotel,600252.45,5340039.78
VIE040,Tram_Stop 40,tram_stop,601433.35,5339999.35
VIE041,Restaurant 41,restaurant,601083.68,5340151.3
VIE042,Hotel 42,hotel,601793.4,5341392.21
VIE043,Tram_Stop 43,tram_stop,601620.66,5340175.98
VIE044,Park 44,park,601591.99,5341561.92
VIE045,Restaurant 45,restaurant,601614.57,5340769.48
VIE046,Hotel 46,hotel,600984.78,5341159.22
VIE047,Cafe 47,cafe,601487.59,5341665.4
VIE048,Tram_Stop 48,tram_stop,601625.47,5341349.43
VIE049,Tram_Stop 49,tram_stop,601048.88,5340913.48
VIE050,Bakery 50,bakery,600364.59,5341421.91
VIE051,Bakery 51,bakery,600785.77,5341139.41
VIE052,Bookshop 52,bookshop,601478.05,5341208.46
VIE053,Tram_Stop 53,tram_stop,600331.99,5340943.06
VIE054,Bookshop 54,bookshop,601604.5,5341295.42
VIE055,Tram_Stop 55,tram_stop,600239.3,5340214.43
VIE056,University 56,university,601795.35,5340317.79
VIE057,Library 57,library,601365.25,5340521.16
VIE058,Hotel 58,hotel,600157.64,5340941.38
VIE059,Park 59,park,600522.42,5340892.83
VIE060,Bookshop 60,bookshop,600320.99,5340768.09
VIE061,Hotel 61,hotel,600292.12,5340060.26
VIE062,Bakery 62,bakery,600460.63,5340849.74
VIE063,Hotel 63,hotel,601376.01,5340305.38
VIE064,Tram_Stop 64,tram_stop,601193.36,5340713.0
VIE065,Bakery 65,bakery,601162.32,5341047.38
VIE066,Museum 66,museum,600855.12,5341722.34
VIE067,Library 67,library,601685.51,5341662.73
VIE068,Bookshop 68,bookshop,600641.28,5341645.39
VIE069,Park 69,park,601810.6,5340326.79
VIE070,Bakery 70,bakery,601515.01,5341347.91
VIE071,Tram_Stop 71,tram_stop,600301.15,5340033.68
VIE072,Tram_Stop 72,tram_stop,600450.51,5340876.08
VIE073,Tram_Stop 73,tram_stop,601193.55,5341552.4
VIE074,Bakery 74,bakery,600253.0,5340936.8
VIE075,Hotel 75,hotel,601731.74,5340460.09
VIE076,Tram_Stop 76,tram_stop,601799.01,5340094.96
VIE077,Restaurant 77,restaurant,600443.38,5340535.32
VIE078,Tram_Stop 78,tram_stop,600396.82,5341280.16
VIE079,Tram_Stop 79,tram_stop,600703.86,5340023.76
VIE080,Restaurant 80,restaurant,601000.91,5340141.85"""

poi_table = pd.read_csv(StringIO(POI_CSV))
pois = gpd.GeoDataFrame(
    poi_table,
    geometry=gpd.points_from_xy(poi_table["x_m"], poi_table["y_m"]),
    crs="EPSG:32633",
)

# The query point is already in the same metre-based coordinate system.
query_point = Point(601030.53, 5340868.86)
topic_category = "cafe"


In [ ]:
# Inspect the supplied dataset before querying it.
print("Number of POIs:", len(pois))
display(pois.head())


In [ ]:
# Check how many records exist in each category.
category_counts = pois["category"].value_counts().sort_index()
display(category_counts)


### Optional visualization - inspect the supplied POIs

This overview is only to help you understand the dataset. It is **not required for submission**.


In [ ]:
# Create one complete optional overview figure.
# Plot all supplied POIs, then mark the query point on the same axes.
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(pois.geometry.x, pois.geometry.y, s=20, alpha=0.7, label="POIs")
ax.scatter([query_point.x], [query_point.y], marker="x", s=100, label="Query point")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Supplied Vienna practice POIs")
ax.set_aspect("equal")
ax.legend()
plt.show()


## B1. Find the topic matches

The POI table already has a `category` field, so you do not need to build a text index here. Keep the IDs of all café records as `cafe_ids`.

> **Guided reference:** Go back to **A3. Apply the text condition**. Focus on the pattern of collecting matching document IDs into a **set**. Here the data are simpler: you can filter the `category` column directly instead of building an inverted index.


In [ ]:
# Your code: create cafe_ids from rows whose category equals topic_category.
# Hint: first filter the rows, then take the poi_id column, then convert it to a set.

# cafe_ids = ...

# When finished, display how many café records you found.
# print("Cafe records:", len(cafe_ids))


## B2. Build the spatial index

Build the spatial index **once**. You will reuse the same `poi_index` for both the 500 m and 750 m queries.

> **Guided reference:** Go back to **A5. Build and query the spatial index**. Look for the line where the GeoDataFrame creates its spatial index with `.sindex`.


In [ ]:
# Your code: build the spatial index for pois.

# poi_index = ...


## B3. Query the 500 m search box

The box construction is supplied. Your job is only to query `poi_index` and retrieve the candidate rows.

> **Guided reference:** Go back to **A5. Build and query the spatial index**. Follow the same two-step pattern: **query the existing index with the search box**, then use the returned row positions to select the candidate rows.


In [ ]:
# Define the first meaning of "near".
radius_500 = 500  # metres

# Build a square that safely contains the 500 m circle.
search_box_500 = box(
    query_point.x - radius_500,
    query_point.y - radius_500,
    query_point.x + radius_500,
    query_point.y + radius_500,
)


In [ ]:
# Your code: query the index and retrieve the candidate rows.
# Use sort=True so the row positions are easier to inspect.

# positions_500 = ...
# candidates_500 = ...


In [ ]:
# Run this after you have created candidates_500.
# It helps you inspect the candidate set before the exact distance check.

# print("500 m box candidates:", len(candidates_500))
# display(candidates_500[["poi_id", "name", "category"]].sort_values("poi_id"))


## B4. Check the exact 500 m distances

Now repeat the exact-distance step from Part A. Keep the exact spatial matches as `nearby_500` and their IDs as `spatial_ids_500`.

> **Guided reference:** Go back to **A4 → Check the exact distance for the candidates** (and the same step reused in **A5**). The pattern is: calculate distance only for the candidates, compare it with the radius, then collect the IDs that remain.


In [ ]:
# Your code: calculate one exact distance for every candidate.
# Store the values in a new distance_m column.

# candidates_500["distance_m"] = ...


In [ ]:
# Your code: keep only candidates at most 500 m from the query point.

# nearby_500 = ...
# spatial_ids_500 = ...


In [ ]:
# Run this after the two cells above.
# The table makes the candidate-to-exact filtering step visible.

# display(
#     candidates_500[["poi_id", "name", "category", "distance_m"]]
#     .sort_values("distance_m")
#     .round({"distance_m": 1})
# )
# print("500 m exact spatial matches:", len(spatial_ids_500))


## B5. Combine the spatial and café conditions

A final result must be both a café and within 500 m.

> **Guided reference:** Go back to **A3 → Combine the text and spatial conditions**. The final IDs come from combining the topic-match ID set with the spatial-match ID set.


In [ ]:
# Your code: intersect the two ID sets.

# result_ids_500 = ...


In [ ]:
# Run this after you have created result_ids_500.
# Retrieve the final matching POIs from the original table.

# final_500 = pois[pois["poi_id"].isin(result_ids_500)]
# display(final_500[["poi_id", "name", "category"]].sort_values("poi_id"))
# print("Final café matches within 500 m:", len(result_ids_500))


### Optional visualization - see the 500 m workflow on the map

Run this **after completing B3-B5**. It shows all POIs, the search box, the exact 500 m circle and the final café matches. It is **not required for submission**.


In [ ]:
# Start with all POIs, then highlight the box candidates.
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(pois.geometry.x, pois.geometry.y, s=18, alpha=0.35, label="All POIs")
ax.scatter(candidates_500.geometry.x, candidates_500.geometry.y, s=35, label="Box candidates")

# Highlight the final café matches.
final_500 = pois[pois["poi_id"].isin(result_ids_500)]
ax.scatter(final_500.geometry.x, final_500.geometry.y, s=70, marker="*", label="Final cafés")

# Add the query point, exact 500 m circle and candidate box.
ax.scatter([query_point.x], [query_point.y], marker="x", s=100, label="Query point")
ax.add_patch(Circle((query_point.x, query_point.y), radius_500,
                    fill=False, linewidth=2, label="500 m radius"))
ax.add_patch(Rectangle(
    (query_point.x-radius_500, query_point.y-radius_500),
    2*radius_500, 2*radius_500,
    fill=False, linestyle="--", linewidth=2, label="Search box"
))

# Finish the optional figure.
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("500 m café query: candidates and final matches")
ax.set_aspect("equal")
ax.legend()
plt.show()


## B6. Change “near” to 750 m and reuse the index

Now change only the radius. **Do not rebuild `poi_index`.**

> **Guided reference:** Go back to **A6. Change the definition of “near” to 7.5 km**. Notice what changes (radius, search box, exact-distance filter) and what stays the same (the dataset, the spatial index, and the topic IDs).


In [ ]:
# Define the second query radius and its larger search box.
radius_750 = 750
search_box_750 = box(
    query_point.x - radius_750,
    query_point.y - radius_750,
    query_point.x + radius_750,
    query_point.y + radius_750,
)


In [ ]:
# Your code: reuse poi_index to get the 750 m box candidates.

# positions_750 = ...
# candidates_750 = ...


In [ ]:
# Your code: repeat the exact-distance step for 750 m.

# candidates_750["distance_m"] = ...
# nearby_750 = ...
# spatial_ids_750 = ...


In [ ]:
# Your code: combine the new spatial IDs with the SAME cafe_ids.

# result_ids_750 = ...


In [ ]:
# Run this after completing B6.
# Compare the size of the new candidate, spatial and final result sets.

# print("750 m box candidates:", len(candidates_750))
# print("750 m exact spatial matches:", len(spatial_ids_750))
# print("750 m final café matches:", len(result_ids_750))
# display(
#     pois[pois["poi_id"].isin(result_ids_750)][["poi_id", "name", "category"]]
#     .sort_values("poi_id")
# )


### Optional visualization - compare the final 500 m and 750 m café results

Run this **after completing B6**. It is only a visual comparison and is **not required for submission**.


In [ ]:
# Retrieve the two final result sets.
final_500 = pois[pois["poi_id"].isin(result_ids_500)]
final_750 = pois[pois["poi_id"].isin(result_ids_750)]

# Plot all POIs and highlight the two final café sets.
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(pois.geometry.x, pois.geometry.y, s=18, alpha=0.25, label="All POIs")
ax.scatter(final_750.geometry.x, final_750.geometry.y, s=55, label="Cafés within 750 m")
ax.scatter(final_500.geometry.x, final_500.geometry.y, s=80, marker="*", label="Cafés within 500 m")

# Add the query point and both exact distance circles.
ax.scatter([query_point.x], [query_point.y], marker="x", s=100, label="Query point")
ax.add_patch(Circle((query_point.x, query_point.y), radius_500, fill=False, linewidth=2, label="500 m"))
ax.add_patch(Circle((query_point.x, query_point.y), radius_750, fill=False, linestyle="--", linewidth=2, label="750 m"))

# Finish the optional comparison figure.
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Changing only the definition of 'near'")
ax.set_aspect("equal")
ax.legend()
plt.show()


## B7. Summarise the two queries

Create a compact two-row table. This is the only summary you need for Part C.

> **Guided reference:** Go back to **A7. Guided result recap** to see the kinds of counts/results that matter. For this table, use the variables you already created for the 500 m and 750 m queries; do not recompute the workflow.


In [ ]:
# Your code: create query_summary with one row for 500 m and one for 750 m.
# Suggested columns:
# radius_m, box_candidates, spatial_matches, final_cafes

# query_summary = pd.DataFrame([
#     {...},
#     {...},
# ])
# display(query_summary)


# Part C - What do your results show?

Answer these three questions using your **Part B outputs**. One or two sentences per answer are enough. Focus on describing and interpreting what you actually obtained.


### 1. What happened across the 500 m query steps?

Report the number of search-box candidates, exact spatial matches and final café matches. Briefly describe how the set became smaller from one step to the next.


**Your answer:**


### 2. What changed when “near” increased from 500 m to 750 m?

Use your summary table or final result lists. Which counts changed, and which additional café result(s), if any, appeared?


**Your answer:**


### 3. Look at one 500 m box candidate that did not remain in the exact 500 m result

Give its ID or name and its distance from the query point. In one sentence, describe what happened to this record between the candidate step and the exact-distance step.


**Your answer:**


## Submit your notebook to Moodle

Submit the notebook with:

- the visible outputs from **Guided Part A**;
- your completed **Part B** code and outputs;
- the three short **Part C** answers.

The **Optional visualization** cells are not required.

Before submitting, restart the kernel and run the required sections from top to bottom. Save the `.ipynb` with its outputs visible, then upload it to the Week 8 lab checkpoint on Moodle.

Because the second half of Week 8 is also used for the project workshop, if you do not finish every Part B task during class, submit the version that shows your progress for the attendance checkpoint and continue the practice afterwards.
